In [35]:
from pathlib import Path
import polars as pl
from dataclasses import dataclass
from typing import Union, Iterable


CATEGORY = "Pet_Supplies"
INPUT_DIR = Path.cwd() / "raw"
OUTPUT_DIR = Path.cwd() / "prepared"

review_path = INPUT_DIR / f"review_{CATEGORY}.jsonl"
meta_path   = INPUT_DIR / f"meta_{CATEGORY}.jsonl"
# train_path  = INPUT_DIR / f"{CATEGORY}.train.csv"
events_path  = INPUT_DIR / f"{CATEGORY}.rating_only.csv"

MIN_SEQUENCE_LENGTH = 5
MAX_SEQUENCE_LENGTH = 100

# Метаданные о товарах
MIN_TITLE_LEN = 50
MIN_DESC_LEN = 500
MIN_RATING_NUMBER = 20  # товары с реальным пользовательским сигналом

META_SCHEMA = [
    "parent_asin",
    "main_category",
    "title",
    "average_rating",
    "rating_number",
    "features",
    "description",
    "price",
    "store",
    'categories'
    ]  # для построения эмббедингов не требуются images, videos, bought_together, details


SEQUENCE_SCHEMA = [
    "user_id",
    "parent_asin",
    "timestamp"
    ]

## Подготовка информации о товарах

In [36]:
def _sample_frac_by_main_category(
    lf: pl.LazyFrame,
    *,
    frac: float,          # например 0.35 = 35% датасета
    seed: int = 42,
    min_per_cat: int = 1,
) -> pl.LazyFrame:
    assert 0 < frac <= 1.0

    # детерминированный "рандом"
    lf = lf.with_columns((pl.col("parent_asin").hash() + pl.lit(seed)).alias("_h"))

    # размер категории
    lf = lf.with_columns(pl.len().over("main_category").alias("_cat_size"))

    # сколько брать из категории
    lf = lf.with_columns(
        pl.when((pl.col("_cat_size") * pl.lit(frac)) < pl.lit(min_per_cat))
          .then(pl.lit(min_per_cat))
          .otherwise((pl.col("_cat_size") * pl.lit(frac)).cast(pl.Int64))
          .alias("_take_n")
    )

    # порядковый номер внутри категории
    lf = lf.with_columns(
        (pl.col("_h").rank("ordinal").over("main_category") - 1).cast(pl.Int64).alias("_rn")
    )

    return (
        lf.filter(pl.col("_rn") < pl.col("_take_n"))
          .drop(["_h", "_cat_size", "_take_n", "_rn"])
    )

def _read_meta_lazy(meta_jsonl: Path) -> pl.LazyFrame:
    return pl.scan_ndjson(meta_jsonl, ignore_errors=True).select([pl.col(c) for c in META_SCHEMA])

def _normalize_meta_types(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.select(
        pl.col("parent_asin").cast(pl.Utf8),
        pl.col("main_category").cast(pl.Utf8),
        pl.col("title").cast(pl.Utf8),
        pl.col("average_rating").cast(pl.Float32),
        pl.col("rating_number").cast(pl.Int32),
        pl.col("features").cast(pl.List(pl.Utf8)),
        pl.col("description").cast(pl.List(pl.Utf8)),
        pl.col("price").cast(pl.Float32),
        pl.col("store").cast(pl.Utf8),
        pl.col("categories").cast(pl.List(pl.Utf8)),
    )

def _add_text_fields(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.with_columns(
        pl.col("description").list.join(" ").fill_null("").alias("description_text"),
        pl.col("features").list.join(" ").fill_null("").alias("features_text"),
        pl.col("categories").list.join(" > ").fill_null("").alias("categories_text"),
    )

def _filter_valid_items(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.filter(
        pl.col("parent_asin").is_not_null() &
        pl.col("features_text").is_not_null() &
        pl.col("title").is_not_null() &
        (pl.col("title").str.len_chars() > MIN_TITLE_LEN) &
        (pl.col("description_text").str.len_chars() > MIN_DESC_LEN) &
        (pl.col("rating_number") >= MIN_RATING_NUMBER)
    )

def _fill_nulls(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.with_columns(
        pl.col("main_category").fill_null(""),
        pl.col("title").fill_null(""),
        pl.col("features_text").fill_null(""),
        pl.col("description_text").fill_null(""),
        pl.col("store").fill_null(""),
        pl.col("average_rating").fill_null(pl.lit(None, dtype=pl.Float32)),
        pl.col("rating_number").fill_null(0),
        pl.col("price").fill_null(pl.lit(None, dtype=pl.Float32)),
        pl.col("categories_text").fill_null(""),
    )

def _add_item_context(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.with_columns(
        pl.concat_str(
            [
                pl.lit("Product: "), pl.col("title"),
                pl.lit("\n\nDescription: "), pl.col("description_text"),
                pl.lit("\n\nFeatures: "), pl.col("features_text"),
                pl.lit("\n\nCategory: "), pl.col("main_category"),
                pl.lit("\n\nStore: "), pl.col("store"),
                pl.lit("\n\nAverage rating: "), pl.col("average_rating").cast(pl.Utf8).fill_null(""),
                pl.lit(", Rating count: "), pl.col("rating_number").cast(pl.Utf8),
                pl.lit("\n\nPrice: "), pl.col("price").cast(pl.Utf8).fill_null(""),
                pl.lit("\n\nCategories: "), pl.col("categories_text").cast(pl.Utf8),
            ]
        ).alias("item_context")
    )
    
def prepare_meta_dataset(
    meta_jsonl: Path,
) -> tuple[pl.DataFrame, pl.Series]:
    """
    Returns:
      item_df: cleaned dataframe with item_context
      valid_items: pl.Series of parent_asin (unique)
    """

    lf = _read_meta_lazy(meta_jsonl)
    lf = _normalize_meta_types(lf)
    lf = _add_text_fields(lf)
    lf = _filter_valid_items(lf)

    # оставляем только нужное
    lf = lf.select(
        "parent_asin",
        "main_category",
        "title",
        "average_rating",
        "rating_number",
        "features_text",
        "description_text",
        "price",
        "store",
        "categories_text",
    )

    lf = _fill_nulls(lf)
    lf = _add_item_context(lf)

    item_df = lf.collect(engine="streaming")

    return item_df

In [37]:
item_df = prepare_meta_dataset(meta_path)

print(f"MIN_RATING_NUMBER = {MIN_RATING_NUMBER}")
print(f"Items after filter: {item_df.height:,}")
print(f"rating_number stats: min={item_df['rating_number'].min()}, "
      f"median={item_df['rating_number'].median():.0f}, "
      f"max={item_df['rating_number'].max()}")
item_df.head()

MIN_RATING_NUMBER = 20
Items after filter: 63,319
rating_number stats: min=20, median=79, max=132311


parent_asin,main_category,title,average_rating,rating_number,features_text,description_text,price,store,categories_text,item_context
str,str,str,f32,i32,str,str,f32,str,str,str
"""B00XJG2SLG""","""Pet Supplies""","""Hurtta Pet Collection 14-Inch …",4.4,166,"""Made from highly durable Neopr…","""Hurtta harnesses are suitable …",24.950001,"""Hurtta""","""Pet Supplies > Dogs > Collars,…","""Product: Hurtta Pet Collection…"
"""B01MQTWB5H""","""Pet Supplies""","""4 Pack - 4 Inch Ring Filter So…",4.4,84,"""Micron filter bags provide exc…","""Micron filter bags provide exc…",15.0,"""Encompass All""","""""","""Product: 4 Pack - 4 Inch Ring …"
"""B00F3JRLYQ""","""Pet Supplies""","""Wysong Optimal Adult Canine Fo…",4.4,75,"""42% Protein With High Levels O…","""For the past 35 years the Wyso…",17.459999,"""Wysong""","""Pet Supplies > Dogs > Food > D…","""Product: Wysong Optimal Adult …"
"""B09J5FJYMJ""","""Pet Supplies""","""Nutramax Dasuquin Joint Health…",4.3,641,"""Joint Health Support for Cats:…","""Dasuquin for Cats Soft Chews i…",9.68,"""Dasuquin""","""Pet Supplies > Cats > Health S…","""Product: Nutramax Dasuquin Joi…"
"""B08WZ1Z9ZF""","""Pet Supplies""","""Batiyeer 4 Strings in 4 Pieces…",4.0,95,"""Reliable to use: due to the qu…","""Features: Remind other animal…",null,"""Batiyeer""","""Pet Supplies > Dogs > Collars,…","""Product: Batiyeer 4 Strings in…"


In [38]:
item_df

parent_asin,main_category,title,average_rating,rating_number,features_text,description_text,price,store,categories_text,item_context
str,str,str,f32,i32,str,str,f32,str,str,str
"""B00XJG2SLG""","""Pet Supplies""","""Hurtta Pet Collection 14-Inch …",4.4,166,"""Made from highly durable Neopr…","""Hurtta harnesses are suitable …",24.950001,"""Hurtta""","""Pet Supplies > Dogs > Collars,…","""Product: Hurtta Pet Collection…"
"""B01MQTWB5H""","""Pet Supplies""","""4 Pack - 4 Inch Ring Filter So…",4.4,84,"""Micron filter bags provide exc…","""Micron filter bags provide exc…",15.0,"""Encompass All""","""""","""Product: 4 Pack - 4 Inch Ring …"
"""B00F3JRLYQ""","""Pet Supplies""","""Wysong Optimal Adult Canine Fo…",4.4,75,"""42% Protein With High Levels O…","""For the past 35 years the Wyso…",17.459999,"""Wysong""","""Pet Supplies > Dogs > Food > D…","""Product: Wysong Optimal Adult …"
"""B09J5FJYMJ""","""Pet Supplies""","""Nutramax Dasuquin Joint Health…",4.3,641,"""Joint Health Support for Cats:…","""Dasuquin for Cats Soft Chews i…",9.68,"""Dasuquin""","""Pet Supplies > Cats > Health S…","""Product: Nutramax Dasuquin Joi…"
"""B08WZ1Z9ZF""","""Pet Supplies""","""Batiyeer 4 Strings in 4 Pieces…",4.0,95,"""Reliable to use: due to the qu…","""Features: Remind other animal…",null,"""Batiyeer""","""Pet Supplies > Dogs > Collars,…","""Product: Batiyeer 4 Strings in…"
…,…,…,…,…,…,…,…,…,…,…
"""B0013ZEGWO""","""Pet Supplies""","""Sera 7112 Koi Royal Mini 1.8 l…",4.6,1054,"""Staple food for Koi Strengthen…","""Sera Koi Royal is the staple f…",20.75,"""Sera""","""Pet Supplies > Fish & Aquatic …","""Product: Sera 7112 Koi Royal M…"
"""B099J628C8""","""""","""Guess What? My Mom is Pregnant…",4.6,161,"""Texture of material：We use 100…","""Trendy Designs: Our goal was t…",10.99,"""Yangmics Direct""","""Pet Supplies > Dogs > Apparel …","""Product: Guess What? My Mom is…"
"""B0BJFS2B7J""","""Pet Supplies""","""DIELLY 5 Set AI Artificial Ins…",4.5,27,"""【Dog AI Artificial Inseminatio…","""5 Set AI Artificial Inseminati…",9.88,"""DIELLY""","""Pet Supplies > Dogs > Training…","""Product: DIELLY 5 Set AI Artif…"


## Подготовка последовательностей

In [39]:
def _read_events_csv(events_path: Path) -> pl.DataFrame:
    df = pl.read_csv(events_path)
    missing = [c for c in SEQUENCE_SCHEMA if c not in df.columns]
    if missing:
        raise ValueError(f"Events file is missing columns: {missing}. Have: {df.columns}")
    return df.select(list(SEQUENCE_SCHEMA))


def _clean_events(events: pl.DataFrame) -> pl.DataFrame:
    """
    - casts
    - drops nulls
    - sorts by (user_id, timestamp)
    """
    return (
        events.with_columns(
            pl.col("user_id").cast(pl.Utf8),
            pl.col("parent_asin").cast(pl.Utf8),
            pl.col("timestamp").cast(pl.Int64),
        )
        .drop_nulls(["user_id", "parent_asin", "timestamp"])
        .sort(["user_id", "timestamp"])
    )


def _filter_events_by_valid_items(
    events: pl.DataFrame,
    valid_items: Union[pl.Series, Iterable[str]],
) -> pl.DataFrame:
    """
    Фильтр событий: оставляем только те parent_asin, которые есть в valid_items.
    Поддерживает pl.Series и любые Iterable (list/set/tuple).
    """
    valid_s = valid_items if isinstance(valid_items, pl.Series) else pl.Series("parent_asin", list(valid_items))
    valid_s = valid_s.cast(pl.Utf8).drop_nulls().unique()
    return events.filter(pl.col("parent_asin").is_in(valid_s.to_list()))


def _build_sequences(events: pl.DataFrame) -> pl.DataFrame:
    """
    Группируем события в последовательности:
      - sequence: List[str] в хронологическом порядке
      - обрезаем до последних max_sequence_length
      - фильтруем по min_sequence_length
    """
    user_seq = (
        events.group_by("user_id", maintain_order=True)
        .agg(pl.col("parent_asin").alias("sequence"))
        .with_columns(pl.col("sequence").list.tail(MAX_SEQUENCE_LENGTH).alias("sequence"))
        .with_columns(pl.col("sequence").list.len().alias("seq_len"))
        .filter(pl.col("seq_len") >= MIN_SEQUENCE_LENGTH)
    )
    return user_seq


def build_user_sequences(
    events_path: Path,
    valid_items: Union[pl.Series, Iterable[str]],
) -> pl.DataFrame:
    """
    Out: user_id | sequence (List[str]) | seq_len
    """
    events = _read_events_csv(events_path)
    events = _clean_events(events)
    events = _filter_events_by_valid_items(events, valid_items)
    return _build_sequences(events)

In [40]:
# Список доступных товаров
valid_items = item_df.get_column("parent_asin").unique()

sequences_df = build_user_sequences(events_path, valid_items)

sequences_df.head()

user_id,sequence,seq_len
str,list[str],u32
"""AE22236AFRRSMQIKGG7TPTB75QEA""","[""B0002C7FHC"", ""B00UFKQKLS"", … ""B09FQDQMNY""]",7
"""AE222MW56PH6JXPIB6XSAMCBTLNQ""","[""B019RS9JHU"", ""B014QBYLXA"", … ""B07R3VK1XS""]",5
"""AE226DXXSDWPBFTQB3M4VMOVZR2A""","[""B09MLKSC3T"", ""B07C2NDSK5"", … ""B07TG1C14B""]",12
"""AE22BSUSOBFTHDDCCXCS3YWJ7LBA""","[""B0C499F7SF"", ""B0C4Z4H3DT"", … ""B09JSFCBWK""]",7
"""AE22BT6AWP25OAIPYV36NWGXMYPA""","[""B0BG6VPJ72"", ""B0BTLHXCQ1"", … ""B0BYF1FH2P""]",6


In [41]:
sequences_df

user_id,sequence,seq_len
str,list[str],u32
"""AE22236AFRRSMQIKGG7TPTB75QEA""","[""B0002C7FHC"", ""B00UFKQKLS"", … ""B09FQDQMNY""]",7
"""AE222MW56PH6JXPIB6XSAMCBTLNQ""","[""B019RS9JHU"", ""B014QBYLXA"", … ""B07R3VK1XS""]",5
"""AE226DXXSDWPBFTQB3M4VMOVZR2A""","[""B09MLKSC3T"", ""B07C2NDSK5"", … ""B07TG1C14B""]",12
"""AE22BSUSOBFTHDDCCXCS3YWJ7LBA""","[""B0C499F7SF"", ""B0C4Z4H3DT"", … ""B09JSFCBWK""]",7
"""AE22BT6AWP25OAIPYV36NWGXMYPA""","[""B0BG6VPJ72"", ""B0BTLHXCQ1"", … ""B0BYF1FH2P""]",6
…,…,…
"""AHZZRNQM6RYH2I2XJXKZ2N4VUR7A""","[""B0B3CWS5J9"", ""B00LP7OQRU"", … ""B08962PMQS""]",5
"""AHZZVRRWWBJYSAGGJGOIP7MLQPYA""","[""B0C1JYQGZM"", ""B001U8L8NO"", … ""B0BXM1HWCY""]",5
"""AHZZWOTTL2PHAINOS7YFVOAQHB2Q""","[""B0792FV1J7"", ""B0BSSSB6T1"", … ""B07P5M27MJ""]",5


## Сохранение

In [42]:
# Save the filtered sequences with full history
output_path = OUTPUT_DIR / f"{CATEGORY}_sequences.parquet"
sequences_df.write_parquet(output_path)
print(f"Saved filtered sequences to: {output_path} (rows = {sequences_df.shape[0]:,})")

# Save the valid items metadata
metadata_output_path = OUTPUT_DIR / f"{CATEGORY}_items.parquet"
item_df.write_parquet(metadata_output_path)
print(f"Saved valid item metadata to: {metadata_output_path} (rows = {len(item_df):,})")

Saved filtered sequences to: /Users/kalistratov/Documents/PYTHON PROJECTS/EDUCATION PROJECTS/MIPT_MASTER/mipt_master/data/prepared/Pet_Supplies_sequences.parquet (rows = 101,926)
Saved valid item metadata to: /Users/kalistratov/Documents/PYTHON PROJECTS/EDUCATION PROJECTS/MIPT_MASTER/mipt_master/data/prepared/Pet_Supplies_items.parquet (rows = 63,319)
